# Checkout the Canteen - Workflow Colab/Kaggle/Drive

Notebook này là entrypoint chính để train project trên **Google Colab**, **Kaggle Notebook**, hoặc chạy smoke test local. Data lớn không nằm trong GitHub; workflow mặc định dùng `classification.zip`, copy về runtime rồi unzip local trước khi train.

## 0. Chọn môi trường

- `auto`: tự nhận Colab/Kaggle/local.
- `colab`: mount Google Drive, tìm `MyDrive/canteen_checkout/datasets/classification.zip`.
- `kaggle`: tìm `classification.zip` hoặc folder `classification` trong `/kaggle/input`.
- `local`: dùng repo đang mở và `data/classification`.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, zipfile, json
from datetime import datetime

ENV = "auto"  # auto | colab | kaggle | local
REPO_URL = "https://github.com/Vo-Minh-Tri1412/cnn-food-recognition.git"
BRANCH = "codex/colab-kaggle-workflow"

def detect_env():
    if ENV != "auto":
        return ENV
    if "COLAB_RELEASE_TAG" in os.environ or Path('/content').exists():
        return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path('/kaggle').exists():
        return "kaggle"
    return "local"

ENV_NAME = detect_env()
print('ENV_NAME =', ENV_NAME)
print('Python =', sys.executable)

## 1. Clone hoặc vào đúng thư mục repo

Khi mở notebook trực tiếp trên Colab/Kaggle, cell này sẽ clone branch workflow từ GitHub. Khi chạy local trong repo, nó giữ nguyên thư mục hiện tại.

In [ ]:
def run(cmd, cwd=None, env=None):
    print('>', ' '.join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True)

def has_repo_files(path: Path) -> bool:
    return (path / 'scripts').exists() and (path / 'canteen_checkout').exists()

if ENV_NAME == 'colab':
    REPO_DIR = Path('/content/cnn-food-recognition')
elif ENV_NAME == 'kaggle':
    REPO_DIR = Path('/kaggle/working/cnn-food-recognition')
else:
    REPO_DIR = Path.cwd()

if ENV_NAME in {'colab', 'kaggle'} and not has_repo_files(REPO_DIR):
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
PROJECT_ROOT = Path.cwd()
print('PROJECT_ROOT =', PROJECT_ROOT)

## 2. Cài dependency tối thiểu

Colab/Kaggle thường đã có PyTorch GPU. Cell này chỉ cài các package phụ để tránh reinstall torch quá nặng.

In [ ]:
if ENV_NAME in {'colab', 'kaggle'}:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python', 'pandas', 'matplotlib', 'scikit-learn', 'tqdm', 'seaborn', 'pyyaml'])
else:
    print('Local mode: bỏ qua pip install. Hãy dùng .venv đã setup.')

## 3. Kiểm tra GPU

In [ ]:
import torch
print('torch =', torch.__version__)
print('cuda =', torch.cuda.is_available())
print('gpu =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 4. Mount Google Drive nếu chạy Colab

Mặc định dataset zip nằm ở:

`MyDrive/canteen_checkout/datasets/classification.zip`

In [ ]:
DRIVE_ROOT = None
DRIVE_DATASET_ZIP = None
DRIVE_RUNS_DIR = None

if ENV_NAME == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/canteen_checkout')
    DRIVE_DATASET_ZIP = DRIVE_ROOT / 'datasets' / 'classification.zip'
    DRIVE_RUNS_DIR = DRIVE_ROOT / 'runs'
    print('DRIVE_DATASET_ZIP =', DRIVE_DATASET_ZIP)
else:
    print('Không phải Colab, bỏ qua mount Drive.')

## 5. Chuẩn bị dataset runtime

Không train trực tiếp trên Drive. Notebook sẽ copy `classification.zip` vào runtime rồi unzip local.

In [ ]:
if ENV_NAME == 'colab':
    WORK_DATA_ROOT = Path('/content/canteen_checkout_data')
elif ENV_NAME == 'kaggle':
    WORK_DATA_ROOT = Path('/kaggle/working/canteen_checkout_data')
else:
    WORK_DATA_ROOT = PROJECT_ROOT / 'data'

CLASSIFICATION_ROOT = WORK_DATA_ROOT / 'classification'
WORK_DATA_ROOT.mkdir(parents=True, exist_ok=True)

def find_kaggle_dataset():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None
    zips = sorted(input_root.rglob('classification.zip'))
    if zips:
        return zips[0]
    folders = [p for p in input_root.rglob('classification') if (p / 'train').exists() and (p / 'val').exists()]
    return sorted(folders)[0] if folders else None

def unzip_dataset(zip_path: Path, target_root: Path):
    local_zip = target_root / 'classification.zip'
    if zip_path.resolve() != local_zip.resolve():
        shutil.copy2(zip_path, local_zip)
    if CLASSIFICATION_ROOT.exists():
        shutil.rmtree(CLASSIFICATION_ROOT)
    with zipfile.ZipFile(local_zip, 'r') as archive:
        archive.extractall(target_root)
    print('Unzipped to', CLASSIFICATION_ROOT)

if ENV_NAME == 'local' and CLASSIFICATION_ROOT.exists():
    print('Dùng local dataset:', CLASSIFICATION_ROOT)
elif ENV_NAME == 'colab':
    if not DRIVE_DATASET_ZIP or not DRIVE_DATASET_ZIP.exists():
        raise FileNotFoundError(f'Không thấy dataset zip trên Drive: {DRIVE_DATASET_ZIP}')
    unzip_dataset(DRIVE_DATASET_ZIP, WORK_DATA_ROOT)
elif ENV_NAME == 'kaggle':
    source = find_kaggle_dataset()
    if source is None:
        raise FileNotFoundError('Không thấy classification.zip hoặc folder classification trong /kaggle/input')
    if source.is_file():
        unzip_dataset(source, WORK_DATA_ROOT)
    else:
        if CLASSIFICATION_ROOT.exists():
            shutil.rmtree(CLASSIFICATION_ROOT)
        shutil.copytree(source, CLASSIFICATION_ROOT)
        print('Copied dataset folder to', CLASSIFICATION_ROOT)
else:
    local_zip = PROJECT_ROOT / 'outputs' / 'cloud' / 'classification.zip'
    if local_zip.exists():
        unzip_dataset(local_zip, WORK_DATA_ROOT)
    else:
        raise FileNotFoundError('Không thấy data/classification hoặc outputs/cloud/classification.zip')

print('CLASSIFICATION_ROOT =', CLASSIFICATION_ROOT)

## 6. Audit dataset

In [ ]:
run([sys.executable, 'scripts/20_audit_dataset_conflicts.py', '--root', CLASSIFICATION_ROOT, '--phash-threshold', '4'])

## 7. Train model

Đổi `EPOCHS = 1` để smoke test nhanh. Khi train thật, dùng `6` đến `10` epoch.

In [ ]:
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
if ENV_NAME == 'colab':
    RUN_ROOT = Path('/content/canteen_checkout_runs') / RUN_ID
elif ENV_NAME == 'kaggle':
    RUN_ROOT = Path('/kaggle/working/canteen_checkout_runs') / RUN_ID
else:
    RUN_ROOT = PROJECT_ROOT / 'outputs' / 'cloud_runs' / RUN_ID

MODEL_DIR = RUN_ROOT / 'models'
OUTPUTS_DIR = RUN_ROOT / 'outputs'
MODEL_OUT = MODEL_DIR / 'dish_classifier.pt'
REPORT_OUT = OUTPUTS_DIR / 'reports'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_OUT.mkdir(parents=True, exist_ok=True)

# Doi MODEL_ARCH de test model manh hon tren Colab GPU.
# Goi y: efficientnet_b0 an toan; efficientnet_b2 can bang; convnext_tiny/resnet50 nang hon.
MODEL_CHOICES = [
    'mobilenet_v3_small', 'mobilenet_v3_large',
    'efficientnet_b0', 'efficientnet_b1', 'efficientnet_b2', 'efficientnet_b3',
    'resnet18', 'resnet50', 'convnext_tiny',
]
MODEL_ARCH = 'efficientnet_b2'
AUGMENTATION = 'strong'
EPOCHS = 6
BATCH_SIZE = 8
IMAGE_SIZE = 224
LR = '0.0001'
assert MODEL_ARCH in MODEL_CHOICES, MODEL_ARCH
print('MODEL_ARCH =', MODEL_ARCH)
print('AUGMENTATION =', AUGMENTATION)

train_env = os.environ.copy()
train_env['CANTEEN_MODEL_DIR'] = str(MODEL_DIR)
train_env['CANTEEN_OUTPUTS_DIR'] = str(OUTPUTS_DIR)

run([
    sys.executable, 'scripts/05_train_classifier.py',
    '--data', CLASSIFICATION_ROOT,
    '--model-out', MODEL_OUT,
    '--arch', MODEL_ARCH,
    '--augmentation', AUGMENTATION,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--image-size', str(IMAGE_SIZE),
    '--lr', LR,
    '--label-smoothing', '0.05',
    '--num-workers', '2' if ENV_NAME in {'colab', 'kaggle'} else '0',
], env=train_env)

print('RUN_ROOT =', RUN_ROOT)
print('MODEL_OUT =', MODEL_OUT)
print('REPORT_OUT =', REPORT_OUT)

## 8. Lưu kết quả về Drive nếu chạy Colab

Kaggle sẽ giữ output trong `/kaggle/working`, có thể tải từ tab Output.

In [ ]:
if ENV_NAME == 'colab':
    DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
    drive_target = DRIVE_RUNS_DIR / RUN_ID
    if drive_target.exists():
        shutil.rmtree(drive_target)
    shutil.copytree(RUN_ROOT, drive_target)
    print('Đã copy run về Drive:', drive_target)
else:
    print('Run output giữ tại:', RUN_ROOT)

## 9. Đóng gói dataset để upload Drive/Kaggle Dataset (chạy local)

Cell này dành cho máy local sau khi bạn đã build `data/classification`. File zip sinh ra ở `outputs/cloud/classification.zip`.

In [ ]:
if ENV_NAME == 'local':
    run([sys.executable, 'scripts/27_package_cloud_dataset.py'])
else:
    print('Bỏ qua package dataset vì đang chạy cloud runtime.')